In [11]:
%load_ext autoreload
%autoreload 2
    
import glob, os, sys
import shutil
import numpy as np
import torch

import pickle

from decimal import Decimal
from itertools import product

import pickle

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
data_dir = "/plp_scr1/agar_sh/data/TPH/"
# define here
data_dir = ""
nn_dir = ""
# or
# import from a file
from paths import *

In [13]:
sims = torch.load(data_dir + "/sims.pt")
paras = []
for i in [120,122,119]:
    paras.append([sims[i][2], sims[i][3], sims[i][4]])

paras += [[0, 1e+6, 1],
          [10, 1e+10, 100], 
          [0, 1e+6*0.8, 1],
          [10*1.1, 1e+10*1.1, 100*1.1]
         ]

#for p in paras:
#    print(p[0], ",",  "{:.5E}".format(Decimal(p[1])), ",", "{:.5E}".format(Decimal(p[2])))

/tmp/ipykernel_706109/3627280074.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sims = torch.load(data_dir + "/sims.pt")


In [15]:
combs =   [
            # raq,fkt,fkp, network, padding, kernel, symmetry, layers, p_pred, filters, "loss", batch size, levels, advect, loss_scale, loss_de, l2, 
            # mode, mmskip, start, urf, solver, Di, write, cool, decay, gpu

            # speedup test
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA", 1,  "hot", 0.99, "iterative", 0.0, 1000, 0, 0, 0],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "hot", 1.0,  "mumps", 0.0, 100, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
    
            # parameter tests
            [0.80523448 , 3.29628E+6 , 6.37660E+0] + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [9.17743012 , 1.79784E+8 , 7.49928E+0] + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [0 , 1.00E+6 , 1.0000E+0]          + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [10 , 1.00E+10 , 1.0000E+2]        + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [0.80523448 , 3.29628E+6 , 6.37660E+0] + [" "]*14 + ["GAIA",  100, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [9.17743012 , 1.79784E+8 , 7.49928E+0] + [" "]*14 + ["GAIA",  100, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [0 , 1.00E+6 , 1.0000E+0]          + [" "]*14 + ["GAIA",  100, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [10 , 1.00E+10 , 1.0000E+2]        + [" "]*14 + ["GAIA",  100, "hot", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [0.80523448 , 3.29628E+6 , 6.37660E+0] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 1],
            [9.17743012 , 1.79784E+8 , 7.49928E+0] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 2],
            [0 , 1.00E+6 , 1.0000E+0]          + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 2],
            [10 , 1.00E+10 , 1.0000E+2]        + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 3],
            [0.80523448 , 3.29628E+6 , 6.37660E+0] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 6],
            [9.17743012 , 1.79784E+8 , 7.49928E+0] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 7],
            [0 , 1.00E+6 , 1.0000E+0]          + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [10 , 1.00E+10 , 1.0000E+2]        + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 1],

            # parameters but with ML
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [0.80523448 , 3.29628E+6 , 6.37660E+0] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [9.17743012 , 1.79784E+8 , 7.49928E+0] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [0 , 1.00E+6 , 1.0000E+0] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                              "ML", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [10 , 1.00E+10 , 1.0000E+2] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                              "ML", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            
            # EBA tests
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1,  "hot", 1.0,  "mumps", 0.1, 1000, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1,  "hot", 0.99,  "iterative", 0.1, 1000, 0, 0, 0],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100,  "hot", 1.0,  "mumps", 0.1, 100, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML_STOKES", int(1e+9),  "hot",  1.0,  "mumps", 0.1, 100, 0, 0, 2],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML_STOKES", int(1e+9),  "hot",  1.0,  "mumps", 0.1, 100, 0, 0, 3],

            # different starts
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1, "cold",    1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1, "linear",  1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1, "perfect", 1.0, "mumps", 0.0, 1000, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "cold",    1.0, "mumps", 0.0, 100, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "linear",  1.0, "mumps", 0.0, 100, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "perfect", 1.0, "mumps", 0.0, 100, 0, 0, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "cold",    1.0,  "mumps", 0.0, 100, 0, 0, 6],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "linear",  1.0,  "mumps", 0.0, 100, 0, 0, 7],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "perfect", 1.0,  "mumps", 0.0, 100, 0, 0, 0],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "cold",    1.0,  "mumps", 0.0, 100, 0, 0, 1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "linear",  1.0,  "mumps", 0.0, 100, 0, 0, 2],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9),  "perfect", 1.0,  "mumps", 0.0, 100, 0, 0, 3],

            # radioactive decay test
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  1, "hot", 1.0, "mumps", 0.0, 1000, 0, 1, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "hot", 1.0, "mumps", 0.0, 100, 0, 1, -1],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 1, 0],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 1, 0],

            
            # ablations
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "mass", 16, 5, 0, 1, 1, 0.0, # soft mass consv
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["fluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 1],             # fluidnet 
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 1, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 2],             # symmetry
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "replicate", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, # replicate padding
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 3],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 0, 0, 0.0,   # loss scale
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 6],    
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "zeros", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, # zeros 16 padding 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 7],
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "zeros", 5, 0, 4, 0, 64, "curl", 8, 5, 0, 1, 1, 0.0, # zeros with same para count
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],
            # to run
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 0, 1, 0.0,   # loss scale - de
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 1],  
            # to run
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 0, 0.0,   # no loss scale + de
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 2],  

            # unets # to run
            # to run
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["unet", "zeros", 5, 0, 3, 0, 16, "curl", 16, 5, 0, 0, 0, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 2], #spc
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["unet", "zeros", 5, 0, 3, 0, 64, "curl", 16, 5, 0, 0, 0, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 3], #sit
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["unet", "learned", 5, 0, 3, 0, 6, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 4], #spc+tricks
            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["unet", "learned", 5, 0, 3, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                  "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 5], #sit+tricks
            
            ]

cntr = 0

for comb in combs:
    raq, fkt, fkp, net, pad, ker, sym, rep, p_pred, fil, loss_type, \
    b, levels, advect, loss_scale, loss_derivative, l2, mode, i, start, urf, solver, \
    Di, write, cool, decay, gpu_num = comb

    if raq<1:
        t = "1.0" 
    elif raq>5:
        t = "0.1"
    else:
        t = "0.2"

    if decay:
        t = "0.056"

    if mode == "GAIA":
         suff = " -w 2 -save 200 -write " + str(write) + " -i " + str(i) + " -init " + start + " -u " + str(urf) \
              + " -sol " + solver + " -di " + str(Di) + " -cool " + str(cool) + " -decay " + str(decay) + " >out_" + str(cntr) + ".txt &;"
    else:
         suff = " -w 2 -i " + str(i) + " -init " + start + " -u " + str(urf) \
              + " -sol " + solver + " -di " + str(Di) + " -gpu " + str(gpu_num) + " -f " + str(fil) + " -s " + str(sym) + " -r " \
              + str(rep) + " -k " + str(ker) + " -lt " + loss_type + " -pp " + str(p_pred) + " -ad 2 " \
              + " -b " + str(b) + " -l " + str(levels) + " -ad_loss " + str(advect) + " -l_sc " + str(loss_scale)  + " -l_de " + str(loss_derivative) \
              + " -l2 "  + str(l2) + " -save 200 -write " + str(write) + " -net " + str(net) + " -pad " + str(pad) \
              + " -cool " + str(cool) + " -decay " + str(decay) \
              + " -e -1 " + " >out_" + str(cntr) + ".txt &;"

    if 1==1: #(sym==0 or sym == " ") and decay: 
        print("python advect_wi_gaia.py -t " + t + " -m " + mode + " -raq " + str(raq) + " -fkt " + str(fkt) + " -fkp " + str(fkp) + suff)

        cntr += 1
    #print()

python advect_wi_gaia.py -t 0.1 -m GAIA -raq 8.75081696 -fkt 1686050000.0 -fkp 28.8237 -w 2 -save 200 -write 1000 -i 1 -init hot -u 1.0 -sol mumps -di 0.0 -cool 0 -decay 0 >out_0.txt &;
python advect_wi_gaia.py -t 0.1 -m GAIA -raq 8.75081696 -fkt 1686050000.0 -fkp 28.8237 -w 2 -save 200 -write 1000 -i 1 -init hot -u 0.99 -sol iterative -di 0.0 -cool 0 -decay 0 >out_1.txt &;
python advect_wi_gaia.py -t 0.1 -m GAIA -raq 8.75081696 -fkt 1686050000.0 -fkp 28.8237 -w 2 -save 200 -write 100 -i 100 -init hot -u 1.0 -sol mumps -di 0.0 -cool 0 -decay 0 >out_2.txt &;
python advect_wi_gaia.py -t 0.1 -m ML_STOKES -raq 8.75081696 -fkt 1686050000.0 -fkp 28.8237 -w 2 -i 1000000000 -init hot -u 1.0 -sol mumps -di 0.0 -gpu 0 -f 16 -s 0 -r 6 -k 5 -lt curl -pp 0 -ad 2  -b 16 -l 5 -ad_loss 0 -l_sc 1 -l_de 1 -l2 0.0 -save 200 -write 100 -net newfluidnet -pad learned -cool 0 -decay 0 -e -1  >out_3.txt &;
python advect_wi_gaia.py -t 1.0 -m GAIA -raq 0.80523448 -fkt 3296280.0 -fkp 6.3766 -w 2 -save 200 -write

In [8]:
combs =   [
            # raq,fkt,fkp, network, padding, kernel, symmetry, layers, p_pred, filters, "loss", batch size, levels, advect, loss_scale, loss_de, l2, 
            # mode, mmskip, start, urf, solver, Di, write, cool, decay, gpu

            #[8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA", 1,  "hot", 0.99, "iterative", 0.0, 1000, 0, 0, 0],

            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "learned", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],

            [8.75081696 , 1.68605E+9 , 2.88237E+1] + ["newfluidnet", "zeros", 5, 0, 6, 0, 16, "curl", 16, 5, 0, 1, 1, 0.0, 
                                                      "ML_STOKES", int(1e+9), "hot", 1.0, "mumps", 0.0, 100, 0, 0, 0],

            [8.75081696 , 1.68605E+9 , 2.88237E+1] + [" "]*14 + ["GAIA",  100, "hot", 1.0,  "mumps", 0.0, 100, 0, 0, -1],
            ]

cntr = 0

gpus = [0,1,2,3,4,5,6,7]*50
 #[0,1,0,1,0,1,2]*10 #
for comb in combs:
    _,_,_, net, pad, ker, sym, rep, p_pred, fil, loss_type, \
    b, levels, advect, loss_scale, loss_derivative, l2, mode, i, start, urf, solver, \
    Di, write, cool, decay, gpu_num = comb
    for sim in sims:
        if sim[1] == "test": # and sim[0] not in [119,120,122]:
            raq, fkt, fkp = sim[2:5]
        
            
            if fkt>5e+8:
                t="0.1"
            else:
                t="1.0"
        
            if decay:
                t = "0.056"

            gpu_num = gpus[cntr]
            if mode == "GAIA":
                 suff = " -w 2 -save 200 -write " + str(write) + " -i " + str(i) + " -init " + start + " -u " + str(urf) \
                      + " -sol " + solver + " -di " + str(Di) + " -cool " + str(cool) + " -decay " + str(decay) + " >out_" + str(cntr+1) + ".txt &;"
            else:
                 suff = " -gpu " + str(gpu_num) + " -w 2 -i " + str(i) + " -init " + start + " -u " + str(urf) \
                      + " -sol " + solver + " -di " + str(Di) + " -f " + str(fil) + " -s " + str(sym) + " -r " \
                      + str(rep) + " -k " + str(ker) + " -lt " + loss_type + " -pp " + str(p_pred) + " -ad 2 " \
                      + " -b " + str(b) + " -l " + str(levels) + " -ad_loss " + str(advect) + " -l_sc " + str(loss_scale)  + " -l_de " + str(loss_derivative) \
                      + " -l2 "  + str(l2) + " -save 200 -write " + str(write) + " -net " + str(net) + " -pad " + str(pad) \
                      + " -cool " + str(cool) + " -decay " + str(decay) \
                      + " -e -1 " + " >out_" + str(cntr+1) + ".txt &;"
            
            print("python advect_wi_gaia.py -t " + t + " -m " + mode + " -raq " + str(raq) + " -fkt " + str(fkt) + " -fkp " + str(fkp) + suff)
        
            cntr += 1
            #print()
    for _ in range(8):
        print()

print(cntr)

python advect_wi_gaia.py -t 0.1 -m ML_STOKES -raq 7.87992683 -fkt 5860425410.0 -fkp 5.70103793 -gpu 0 -w 2 -i 1000000000 -init hot -u 1.0 -sol mumps -di 0.0 -f 16 -s 0 -r 6 -k 5 -lt curl -pp 0 -ad 2  -b 16 -l 5 -ad_loss 0 -l_sc 1 -l_de 1 -l2 0.0 -save 200 -write 100 -net newfluidnet -pad learned -cool 0 -decay 0 -e -1  >out_1.txt &;
python advect_wi_gaia.py -t 1.0 -m ML_STOKES -raq 0.26710223 -fkt 3636478.69 -fkp 3.57980514 -gpu 1 -w 2 -i 1000000000 -init hot -u 1.0 -sol mumps -di 0.0 -f 16 -s 0 -r 6 -k 5 -lt curl -pp 0 -ad 2  -b 16 -l 5 -ad_loss 0 -l_sc 1 -l_de 1 -l2 0.0 -save 200 -write 100 -net newfluidnet -pad learned -cool 0 -decay 0 -e -1  >out_2.txt &;
python advect_wi_gaia.py -t 1.0 -m ML_STOKES -raq 2.22284414 -fkt 10277545.5 -fkp 1.08850972 -gpu 2 -w 2 -i 1000000000 -init hot -u 1.0 -sol mumps -di 0.0 -f 16 -s 0 -r 6 -k 5 -lt curl -pp 0 -ad 2  -b 16 -l 5 -ad_loss 0 -l_sc 1 -l_de 1 -l2 0.0 -save 200 -write 100 -net newfluidnet -pad learned -cool 0 -decay 0 -e -1  >out_3.txt &;